# Coding-task data exploration for ICMLW26-style compositional self-improvement

This notebook is a **data and task-shape investigation**, not a training pipeline. It compares the two ideas in `new_task_candidates/coding`:

- **BFCL-Comp:** one function call → two independent calls → four independent calls.
- **CommitPack-Config:** one configuration edit → two independent edits → four independent edits.

The setup is deliberately the manual curriculum from the ICMLW26 paper: fixed rounds, fixed task rules, supervised fine-tuning in a future implementation, and **no reinforcement learning or model-proposed curricula**.

What this notebook does:

1. downloads bounded, pinned official data;
2. displays raw records and exploratory profiles;
3. previews seed-train, composed-train, and evaluation example shapes;
4. lists illustrative guard accept/reject cases; and
5. surfaces questions to answer before building a serious pipeline.

The composed targets below use hidden/oracle references only to illustrate dataset shape. They are **not** pseudo-labels produced by a model.


In [ ]:
from __future__ import annotations

import hashlib
import html
import importlib.metadata
import json
import math
import os
from pathlib import Path
from typing import Any

import jsonpatch
import matplotlib.pyplot as plt
import pandas as pd
import requests
import yaml
from IPython.display import HTML, Markdown, display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "new_task_candidates" / "coding").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current working directory.")

ROOT = find_repo_root()
BFCL_REVISION = "61fc0608cfd831fcfbbaa676ebdfef0ed963eeda"
COMMITPACK_REVISION = "fc56fe33c030c6daa414c2b112c932b8eed085e6"
COMMITPACK_SCAN_ROWS = int(os.getenv("CODING_DATA_SCAN_ROWS", "500"))
OFFLINE = os.getenv("CODING_DATA_OFFLINE", "0") == "1"
REFRESH = os.getenv("CODING_DATA_REFRESH", "0") == "1"
CACHE_DIR = Path(
    os.getenv(
        "CODING_DATA_CACHE_DIR",
        str(ROOT / "artifacts" / "data" / "coding_task_data_exploration"),
    )
).expanduser().resolve()
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Cache directory: {CACHE_DIR}")
print(f"Offline={OFFLINE}, refresh={REFRESH}, CommitPack scan rows/language={COMMITPACK_SCAN_ROWS}")


## Data sources and reproducibility

The notebook pins source revisions instead of following a moving `main` branch.

- BFCL: the non-live `simple`, `parallel`, and `parallel_multiple` categories plus their possible-answer files. The filenames retain `BFCL_v3_*`, but these are the single-turn categories used by the BFCL-Comp idea—not agentic or multi-turn tasks.
- CommitPackFT: a bounded prefix of the JSON and YAML JSONL files. Prefix statistics are convenient for exploration but are **not representative full-dataset estimates**.

Downloads are cached under ignored `artifacts/`. Set `CODING_DATA_OFFLINE=1` to use an existing cache or the embedded fallback examples.


In [ ]:
BFCL_FALLBACK = {
    "simple": {
        "questions": [
            {
                "id": "demo_simple_triangle",
                "question": [[{"role": "user", "content": "Find the area of a triangle with base 10 and height 5."}]],
                "function": [{
                    "name": "calculate_triangle_area",
                    "description": "Calculate a triangle area.",
                    "parameters": {
                        "type": "dict",
                        "properties": {"base": {"type": "integer"}, "height": {"type": "integer"}},
                        "required": ["base", "height"],
                    },
                }],
            },
            {
                "id": "demo_simple_factorial",
                "question": [[{"role": "user", "content": "Calculate the factorial of 5."}]],
                "function": [{
                    "name": "math.factorial",
                    "description": "Calculate a factorial.",
                    "parameters": {
                        "type": "dict",
                        "properties": {"number": {"type": "integer"}},
                        "required": ["number"],
                    },
                }],
            },
            {
                "id": "demo_simple_weather",
                "question": [[{"role": "user", "content": "Get the weather in Paris."}]],
                "function": [{
                    "name": "get_weather",
                    "description": "Get weather for a city.",
                    "parameters": {
                        "type": "dict",
                        "properties": {"city": {"type": "string"}},
                        "required": ["city"],
                    },
                }],
            },
            {
                "id": "demo_simple_currency",
                "question": [[{"role": "user", "content": "Convert 100 USD to EUR."}]],
                "function": [{
                    "name": "convert_currency",
                    "description": "Convert an amount between currencies.",
                    "parameters": {
                        "type": "dict",
                        "properties": {
                            "amount": {"type": "integer"},
                            "source": {"type": "string"},
                            "target": {"type": "string"},
                        },
                        "required": ["amount", "source", "target"],
                    },
                }],
            },
        ],
        "answers": [
            {"id": "demo_simple_triangle", "ground_truth": [{"calculate_triangle_area": {"base": [10], "height": [5]}}]},
            {"id": "demo_simple_factorial", "ground_truth": [{"math.factorial": {"number": [5]}}]},
            {"id": "demo_simple_weather", "ground_truth": [{"get_weather": {"city": ["Paris"]}}]},
            {"id": "demo_simple_currency", "ground_truth": [{"convert_currency": {"amount": [100], "source": ["USD"], "target": ["EUR"]}}]},
        ],
    },
    "parallel": {
        "questions": [{
            "id": "demo_parallel",
            "question": [[{"role": "user", "content": "Get the weather in Paris and Rome."}]],
            "function": [{
                "name": "get_weather",
                "description": "Get weather for a city.",
                "parameters": {
                    "type": "dict",
                    "properties": {"city": {"type": "string"}},
                    "required": ["city"],
                },
            }],
        }],
        "answers": [{
            "id": "demo_parallel",
            "ground_truth": [
                {"get_weather": {"city": ["Paris"]}},
                {"get_weather": {"city": ["Rome"]}},
            ],
        }],
    },
    "parallel_multiple": {
        "questions": [{
            "id": "demo_parallel_multiple",
            "question": [[{"role": "user", "content": "Get Paris weather and convert 100 USD to EUR."}]],
            "function": [
                {
                    "name": "get_weather",
                    "description": "Get weather for a city.",
                    "parameters": {"type": "dict", "properties": {"city": {"type": "string"}}, "required": ["city"]},
                },
                {
                    "name": "convert_currency",
                    "description": "Convert currencies.",
                    "parameters": {
                        "type": "dict",
                        "properties": {
                            "amount": {"type": "integer"},
                            "source": {"type": "string"},
                            "target": {"type": "string"},
                        },
                        "required": ["amount", "source", "target"],
                    },
                },
            ],
        }],
        "answers": [{
            "id": "demo_parallel_multiple",
            "ground_truth": [
                {"get_weather": {"city": ["Paris"]}},
                {"convert_currency": {"amount": [100], "source": ["USD"], "target": ["EUR"]}},
            ],
        }],
    },
}

COMMITPACK_FALLBACK = {
    "json": [
        {
            "commit": "demo-json-one",
            "old_file": "app.json",
            "new_file": "app.json",
            "old_contents": json.dumps({"service": {"port": 8080}}, indent=2),
            "new_contents": json.dumps({"service": {"port": 9090}}, indent=2),
            "subject": "Update the service port",
            "message": "Update the service port",
            "lang": "JSON",
            "license": "mit",
            "repos": "demo/example",
        },
        {
            "commit": "demo-json-four",
            "old_file": "config.json",
            "new_file": "config.json",
            "old_contents": json.dumps({
                "service": {"port": 8080},
                "logging": {"level": "info"},
                "features": {"legacy": True},
            }, indent=2),
            "new_contents": json.dumps({
                "service": {"port": 9090, "timeout": 30},
                "logging": {"level": "debug"},
                "features": {},
            }, indent=2),
            "subject": "Refresh service and logging configuration",
            "message": "Refresh service and logging configuration",
            "lang": "JSON",
            "license": "apache-2.0",
            "repos": "demo/example",
        },
    ],
    "yaml": [
        {
            "commit": "demo-yaml-two",
            "old_file": "deploy.yaml",
            "new_file": "deploy.yaml",
            "old_contents": "spec:\n  replicas: 1\n  image: nginx:1.26\n",
            "new_contents": "spec:\n  replicas: 3\n  image: nginx:1.27\n",
            "subject": "Scale and update the deployment",
            "message": "Scale and update the deployment",
            "lang": "YAML",
            "license": "mit",
            "repos": "demo/deployment",
        },
    ],
}

def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()

def stable_rows_hash(rows: list[dict[str, Any]]) -> str:
    payload = json.dumps(rows, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return sha256_bytes(payload)

def read_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def fetch_jsonl(
    *,
    label: str,
    url: str,
    destination: Path,
    fallback_rows: list[dict[str, Any]],
    limit: int | None = None,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    if destination.exists() and (not REFRESH or OFFLINE):
        rows = read_jsonl(destination)
        return rows, {
            "label": label,
            "source": "cache",
            "url": url,
            "path": str(destination),
            "rows": len(rows),
            "sha256": sha256_bytes(destination.read_bytes()),
            "error": None,
        }

    if OFFLINE:
        rows = fallback_rows
        return rows, {
            "label": label,
            "source": "embedded fallback",
            "url": url,
            "path": None,
            "rows": len(rows),
            "sha256": stable_rows_hash(rows),
            "error": "offline mode and no cache",
        }

    temporary = destination.with_suffix(destination.suffix + ".part")
    try:
        destination.parent.mkdir(parents=True, exist_ok=True)
        row_count = 0
        with requests.get(url, stream=True, timeout=(10, 120)) as response:
            response.raise_for_status()
            with temporary.open("wb") as handle:
                for raw_line in response.iter_lines():
                    if not raw_line:
                        continue
                    handle.write(raw_line + b"\n")
                    row_count += 1
                    if limit is not None and row_count >= limit:
                        break
        temporary.replace(destination)
        rows = read_jsonl(destination)
        return rows, {
            "label": label,
            "source": "download",
            "url": url,
            "path": str(destination),
            "rows": len(rows),
            "sha256": sha256_bytes(destination.read_bytes()),
            "error": None,
        }
    except Exception as exc:
        if temporary.exists():
            temporary.unlink()
        rows = fallback_rows
        return rows, {
            "label": label,
            "source": "embedded fallback",
            "url": url,
            "path": None,
            "rows": len(rows),
            "sha256": stable_rows_hash(rows),
            "error": f"{type(exc).__name__}: {exc}",
        }

def show_preformatted(title: str, value: Any, *, max_chars: int = 4000) -> None:
    if not isinstance(value, str):
        value = json.dumps(value, indent=2, ensure_ascii=False, default=str)
    if len(value) > max_chars:
        value = value[:max_chars] + "\n… [truncated]"
    display(Markdown(f"#### {title}"))
    display(HTML(f"<pre style='white-space:pre-wrap'>{html.escape(value)}</pre>"))


In [ ]:
bfcl_data: dict[str, dict[str, list[dict[str, Any]]]] = {}
source_manifest: list[dict[str, Any]] = []

bfcl_base = (
    "https://huggingface.co/datasets/"
    f"gorilla-llm/Berkeley-Function-Calling-Leaderboard/resolve/{BFCL_REVISION}"
)
for category in ("simple", "parallel", "parallel_multiple"):
    question_rows, question_meta = fetch_jsonl(
        label=f"BFCL {category} questions",
        url=f"{bfcl_base}/BFCL_v3_{category}.json",
        destination=CACHE_DIR / BFCL_REVISION / f"BFCL_v3_{category}.jsonl",
        fallback_rows=BFCL_FALLBACK[category]["questions"],
    )
    answer_rows, answer_meta = fetch_jsonl(
        label=f"BFCL {category} possible answers",
        url=f"{bfcl_base}/possible_answer/BFCL_v3_{category}.json",
        destination=CACHE_DIR / BFCL_REVISION / "possible_answer" / f"BFCL_v3_{category}.jsonl",
        fallback_rows=BFCL_FALLBACK[category]["answers"],
    )
    bfcl_data[category] = {"questions": question_rows, "answers": answer_rows}
    source_manifest.extend([question_meta, answer_meta])

commitpack_data: dict[str, list[dict[str, Any]]] = {}
commitpack_base = (
    "https://huggingface.co/datasets/"
    f"bigcode/commitpackft/resolve/{COMMITPACK_REVISION}/data"
)
for language in ("json", "yaml"):
    rows, metadata = fetch_jsonl(
        label=f"CommitPackFT {language} prefix",
        url=f"{commitpack_base}/{language}/data.jsonl",
        destination=(
            CACHE_DIR
            / COMMITPACK_REVISION
            / f"commitpack_{language}_first_{COMMITPACK_SCAN_ROWS}.jsonl"
        ),
        fallback_rows=COMMITPACK_FALLBACK[language],
        limit=COMMITPACK_SCAN_ROWS,
    )
    commitpack_data[language] = rows
    source_manifest.append(metadata)

manifest_payload = {
    "bfcl_revision": BFCL_REVISION,
    "commitpack_revision": COMMITPACK_REVISION,
    "commitpack_scan_rows_per_language": COMMITPACK_SCAN_ROWS,
    "offline": OFFLINE,
    "sources": source_manifest,
}
manifest_path = CACHE_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest_payload, indent=2), encoding="utf-8")

display(pd.DataFrame(source_manifest)[["label", "source", "rows", "sha256", "error"]])
print(f"Runtime manifest: {manifest_path}")


## 1. BFCL-Comp

### Raw shape and reference ambiguity

BFCL possible answers often encode several accepted values for an argument. An exact serialized SFT target is therefore not always uniquely determined without an explicit canonicalization policy. The profile below counts rows where every argument has exactly one accepted value; only those rows are used for the illustrative composed targets.


In [ ]:
def bfcl_question_text(row: dict[str, Any]) -> str:
    messages = row.get("question", [])
    contents: list[str] = []
    for conversation in messages:
        if isinstance(conversation, list):
            for message in conversation:
                if isinstance(message, dict) and message.get("content"):
                    contents.append(str(message["content"]))
    return "\n".join(contents)

def bfcl_answer_is_unique(answer: dict[str, Any]) -> bool:
    ground_truth = answer.get("ground_truth")
    if not isinstance(ground_truth, list) or not ground_truth:
        return False
    for call in ground_truth:
        if not isinstance(call, dict) or len(call) != 1:
            return False
        arguments = next(iter(call.values()))
        if not isinstance(arguments, dict):
            return False
        for accepted_values in arguments.values():
            if not isinstance(accepted_values, list) or len(accepted_values) != 1:
                return False
    return True

def canonical_bfcl_calls(answer: dict[str, Any]) -> list[dict[str, Any]]:
    if not bfcl_answer_is_unique(answer):
        raise ValueError("Reference is not uniquely canonicalizable.")
    calls = []
    for raw_call in answer["ground_truth"]:
        name, argument_options = next(iter(raw_call.items()))
        calls.append({
            "name": name,
            "arguments": {key: values[0] for key, values in argument_options.items()},
        })
    return calls

bfcl_profiles = []
bfcl_answer_lookup: dict[str, dict[str, dict[str, Any]]] = {}
for category, payload in bfcl_data.items():
    questions = payload["questions"]
    answers = payload["answers"]
    answer_by_id = {row["id"]: row for row in answers}
    bfcl_answer_lookup[category] = answer_by_id
    paired = [(row, answer_by_id[row["id"]]) for row in questions if row.get("id") in answer_by_id]
    function_counts = [len(row.get("function", [])) for row, _ in paired]
    required_counts = [
        len(function.get("parameters", {}).get("required", []))
        for row, _ in paired
        for function in row.get("function", [])
    ]
    bfcl_profiles.append({
        "category": category,
        "questions": len(questions),
        "answers": len(answers),
        "paired_ids": len(paired),
        "unique_reference_rows": sum(bfcl_answer_is_unique(answer) for _, answer in paired),
        "mean_function_docs": round(sum(function_counts) / max(len(function_counts), 1), 2),
        "mean_required_args": round(sum(required_counts) / max(len(required_counts), 1), 2),
        "mean_reference_calls": round(
            sum(len(answer.get("ground_truth", [])) for _, answer in paired) / max(len(paired), 1),
            2,
        ),
    })

bfcl_profile_df = pd.DataFrame(bfcl_profiles)
display(bfcl_profile_df)


In [ ]:
simple_row = bfcl_data["simple"]["questions"][0]
simple_answer = bfcl_answer_lookup["simple"][simple_row["id"]]

show_preformatted("Raw BFCL question record", simple_row)
show_preformatted("Aligned possible-answer record", simple_answer)
show_preformatted("User-facing question", bfcl_question_text(simple_row))


### Manual curriculum preview

The examples below deliberately expose oracle targets so that we can inspect formats. A real self-improvement run would keep those references in an audit-only store and would construct frontier targets from current-model component predictions.


In [ ]:
def collect_clean_bfcl_atoms(limit: int = 4) -> list[dict[str, Any]]:
    atoms = []
    used_function_names: set[str] = set()
    for row in bfcl_data["simple"]["questions"]:
        answer = bfcl_answer_lookup["simple"].get(row.get("id"))
        if answer is None or not bfcl_answer_is_unique(answer):
            continue
        functions = row.get("function", [])
        names = {function.get("name") for function in functions}
        if not names or names & used_function_names:
            continue
        atom = {
            "source_id": row["id"],
            "question": bfcl_question_text(row),
            "functions": functions,
            "oracle_target": canonical_bfcl_calls(answer),
        }
        atoms.append(atom)
        used_function_names.update(name for name in names if name)
        if len(atoms) >= limit:
            return atoms
    raise RuntimeError(f"Needed {limit} clean BFCL atoms but found only {len(atoms)}.")

def compose_bfcl_atoms(atoms: list[dict[str, Any]]) -> dict[str, Any]:
    clauses = [atom["question"].rstrip(" .") for atom in atoms]
    prompt = "Complete all requests independently:\n" + "\n".join(
        f"{index}. {clause}" for index, clause in enumerate(clauses, start=1)
    )
    return {
        "source_ids": [atom["source_id"] for atom in atoms],
        "question": prompt,
        "functions": [function for atom in atoms for function in atom["functions"]],
        "oracle_target": [call for atom in atoms for call in atom["oracle_target"]],
    }

bfcl_atoms = collect_clean_bfcl_atoms(4)
bfcl_examples = {
    "Round 0: one-call seed": bfcl_atoms[0],
    "Round 1: two-call oracle-shaped preview": compose_bfcl_atoms(bfcl_atoms[:2]),
    "Round 2: four-call oracle-shaped preview": compose_bfcl_atoms(bfcl_atoms),
}

for title, example in bfcl_examples.items():
    display(Markdown(f"### {title}"))
    show_preformatted("Input question", example["question"])
    display(pd.DataFrame([
        {
            "function": function.get("name"),
            "required": function.get("parameters", {}).get("required", []),
            "properties": list(function.get("parameters", {}).get("properties", {})),
        }
        for function in example["functions"]
    ]))
    show_preformatted("Target JSON list", example["oracle_target"])


In [ ]:
bfcl_eval_rows = []
for category in ("parallel", "parallel_multiple"):
    row = bfcl_data[category]["questions"][0]
    answer = bfcl_answer_lookup[category].get(row["id"], {})
    bfcl_eval_rows.append({
        "category": category,
        "id": row["id"],
        "question": bfcl_question_text(row),
        "function_docs": len(row.get("function", [])),
        "reference_calls": len(answer.get("ground_truth", [])),
        "unique_reference": bfcl_answer_is_unique(answer),
    })
display(pd.DataFrame(bfcl_eval_rows))

bfcl_setting = pd.DataFrame([
    {"stage": "Round 0", "calls": 1, "data role": "labeled seed + replay", "source": "Simple train pool"},
    {"stage": "Round 1", "calls": 2, "data role": "new pseudo-train frontier", "source": "fixed pairs from hidden composition pool"},
    {"stage": "Round 2", "calls": 4, "data role": "new pseudo-train frontier", "source": "fixed pairs of two-call subproblems"},
    {"stage": "Final eval", "calls": "1/2/4/8", "data role": "retention + frontier", "source": "held-out sources/templates/schemas"},
    {"stage": "External eval", "calls": "natural", "data role": "BFCL-derived transfer", "source": "Parallel + Parallel Multiple"},
])
display(bfcl_setting)


### BFCL guard ladder: illustrative cases only

These rows state intended outcomes. The notebook does not implement or certify a BFCL validator.


In [ ]:
bfcl_guard_cases = pd.DataFrame([
    {
        "gate": "G0 — no filtering",
        "expected": "accept",
        "case": "Concatenate two parseable component lists.",
        "why": "Unfiltered composition baseline; semantic/schema mistakes remain.",
    },
    {
        "gate": "G1 — JSON and shape",
        "expected": "reject",
        "case": 'Extra prose followed by {"name": "get_weather", ...}',
        "why": "Target must be a JSON list with call objects only.",
    },
    {
        "gate": "G2 — membership/count",
        "expected": "reject",
        "case": "Two-clause prompt produces one call or names an undocumented function.",
        "why": "Expected call count and available function documents do not match.",
    },
    {
        "gate": "G3 — arguments/types",
        "expected": "reject",
        "case": 'get_weather(arguments={"city": 17})',
        "why": "The required city argument is present but has the wrong schema type.",
    },
    {
        "gate": "G3 — arguments/types",
        "expected": "reject",
        "case": 'convert_currency(arguments={"amount": 100, "source": "USD"})',
        "why": "A required target-currency argument is missing.",
    },
    {
        "gate": "G4 — cross-component",
        "expected": "reject",
        "case": "Two components yield the same exact call when repetition was not requested.",
        "why": "Likely duplicated supervision rather than two independent requests.",
    },
    {
        "gate": "G4 — cross-component",
        "expected": "reject",
        "case": "Clause 2 asks to use the value returned by clause 1.",
        "why": "The requested calls are dependent and cannot be safely flattened.",
    },
    {
        "gate": "G4 — cross-component",
        "expected": "accept",
        "case": "Weather(Paris) plus currency conversion with distinct documented schemas.",
        "why": "Both calls are individually valid and independent.",
    },
    {
        "gate": "G5 — executable (optional)",
        "expected": "reject",
        "case": "A structurally valid executable call fails in the official sandbox.",
        "why": "Optional behavioral safety check; hidden gold output is still not consulted.",
    },
])
display(bfcl_guard_cases)


## 2. CommitPack-Config

### Raw rows and exploratory structural diffs

CommitPackFT supplies before/after file contents and commit metadata, not ready-made JSON Patch tasks. For exploration, the notebook safely parses JSON/YAML and asks `jsonpatch.make_patch` for a structural-diff preview. That preview is library-dependent and is not a proposed canonical production diff.


In [ ]:
def parse_config_document(text: str, language: str) -> Any:
    text = text.lstrip("\ufeff")
    if language == "json":
        value = json.loads(text)
    elif language == "yaml":
        value = yaml.safe_load(text)
    else:
        raise ValueError(f"Unsupported language: {language}")
    if not isinstance(value, (dict, list)):
        raise ValueError("Top-level value is not a mapping or list.")
    # Reject YAML-only objects, non-string keys, dates, and non-finite values.
    encoded = json.dumps(value, ensure_ascii=False, allow_nan=False)
    normalized = json.loads(encoded)
    if normalized != value:
        raise ValueError("Document is not cleanly JSON-compatible.")
    return normalized

commitpack_profiles: list[dict[str, Any]] = []
commitpack_parsed: list[dict[str, Any]] = []
for language, rows in commitpack_data.items():
    for row_index, row in enumerate(rows):
        profile = {
            "language": language,
            "row_index": row_index,
            "commit": row.get("commit"),
            "repo": row.get("repos"),
            "license": row.get("license"),
            "subject": row.get("subject", ""),
            "subject_chars": len(row.get("subject", "")),
            "old_chars": len(row.get("old_contents", "")),
            "new_chars": len(row.get("new_contents", "")),
        }
        try:
            old_document = parse_config_document(row["old_contents"], language)
            new_document = parse_config_document(row["new_contents"], language)
            patch = jsonpatch.make_patch(old_document, new_document).patch
            profile.update({
                "status": "ok" if patch else "no normalized change",
                "operation_count": len(patch),
                "operations": ",".join(sorted({operation.get("op", "?") for operation in patch})),
            })
            if patch:
                commitpack_parsed.append({
                    "language": language,
                    "row": row,
                    "old_document": old_document,
                    "new_document": new_document,
                    "patch": patch,
                })
        except Exception as exc:
            profile.update({
                "status": f"rejected: {type(exc).__name__}",
                "operation_count": math.nan,
                "operations": "",
            })
        commitpack_profiles.append(profile)

commitpack_profile_df = pd.DataFrame(commitpack_profiles)
display(
    commitpack_profile_df.groupby(["language", "status"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
)

valid_commitpack_df = commitpack_profile_df[commitpack_profile_df["status"] == "ok"].copy()
op_count_table = (
    valid_commitpack_df.groupby(["language", "operation_count"])
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["language", "operation_count"])
)
display(op_count_table.head(30))

license_table = (
    commitpack_profile_df.groupby(["language", "license"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["language", "rows"], ascending=[True, False])
)
display(license_table.head(20))


In [ ]:
plot_df = valid_commitpack_df[valid_commitpack_df["operation_count"].between(1, 8)]
if not plot_df.empty:
    pivot = (
        plot_df.groupby(["operation_count", "language"])
        .size()
        .unstack(fill_value=0)
        .sort_index()
    )
    ax = pivot.plot.bar(figsize=(8, 3.5), color={"json": "#4C78A8", "yaml": "#F58518"})
    ax.set_title(f"Patch-size distribution in the bounded first-{COMMITPACK_SCAN_ROWS} sample")
    ax.set_xlabel("Exploratory JSON Patch operation count")
    ax.set_ylabel("Rows")
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()


In [ ]:
raw_commitpack = commitpack_data["json"][0]
show_preformatted(
    "Raw CommitPack metadata",
    {key: raw_commitpack.get(key) for key in ("commit", "old_file", "new_file", "subject", "lang", "license", "repos")},
)
show_preformatted("Old contents", raw_commitpack.get("old_contents", ""), max_chars=2500)
show_preformatted("New contents", raw_commitpack.get("new_contents", ""), max_chars=2500)

matching_preview = next(
    (
        item for item in commitpack_parsed
        if item["row"].get("commit") == raw_commitpack.get("commit")
        and item["language"] == "json"
    ),
    None,
)
if matching_preview is not None:
    show_preformatted("Exploratory structural patch", matching_preview["patch"])


### Manual curriculum preview

The preferred composite source is a single real commit containing several structural operations over one base document. The cells select exact one-, two-, and four-operation rows when available. Controlled instruction text is intentionally simple and is shown only to expose the proposed task format.


In [ ]:
def pick_commitpack_example(operation_count: int) -> dict[str, Any]:
    allowed = {"add", "remove", "replace"}
    exact = [
        item for item in commitpack_parsed
        if len(item["patch"]) == operation_count
        and all(operation.get("op") in allowed for operation in item["patch"])
    ]
    if exact:
        return exact[0]
    larger = [
        item for item in commitpack_parsed
        if len(item["patch"]) >= operation_count
        and all(operation.get("op") in allowed for operation in item["patch"][:operation_count])
    ]
    if larger:
        selected = dict(larger[0])
        selected["patch"] = selected["patch"][:operation_count]
        selected["subset_preview"] = True
        return selected
    raise RuntimeError(f"No {operation_count}-operation CommitPack preview is available.")

def describe_patch_operation(operation: dict[str, Any]) -> str:
    op = operation.get("op")
    path = operation.get("path", "")
    if op == "remove":
        return f"Remove the value at {path}."
    value = json.dumps(operation.get("value"), ensure_ascii=False)
    verb = "Add" if op == "add" else "Set"
    return f"{verb} the value at {path} to {value}."

commitpack_curriculum = {
    "Round 0: one-edit seed": pick_commitpack_example(1),
    "Round 1: two-edit oracle-shaped preview": pick_commitpack_example(2),
    "Round 2: four-edit oracle-shaped preview": pick_commitpack_example(4),
}

for title, item in commitpack_curriculum.items():
    row = item["row"]
    display(Markdown(f"### {title}"))
    display(pd.DataFrame([{
        "language": item["language"],
        "commit": row.get("commit"),
        "repository": row.get("repos"),
        "license": row.get("license"),
        "original subject": row.get("subject"),
        "subset preview": item.get("subset_preview", False),
    }]))
    instructions = "\n".join(
        f"{index}. {describe_patch_operation(operation)}"
        for index, operation in enumerate(item["patch"], start=1)
    )
    show_preformatted("Configuration context (normalized old document)", item["old_document"], max_chars=3500)
    show_preformatted("Controlled requested changes", instructions)
    show_preformatted("Target JSON Patch", item["patch"])


In [ ]:
commitpack_setting = pd.DataFrame([
    {"stage": "Round 0", "edits": 1, "data role": "labeled seed + replay", "source": "atomic operation bank"},
    {"stage": "Round 1", "edits": 2, "data role": "new pseudo-train frontier", "source": "fixed same-commit two-operation subsets"},
    {"stage": "Round 2", "edits": 4, "data role": "new pseudo-train frontier", "source": "fixed pairs of two-edit subproblems"},
    {"stage": "Final eval", "edits": "1/2/4/8", "data role": "retention + frontier", "source": "repository/template-disjoint controlled tasks"},
    {"stage": "Natural eval", "edits": "varied", "data role": "secondary transfer", "source": "aggressively filtered original subjects"},
    {"stage": "Rejected slices", "edits": "varied", "data role": "interaction diagnostic", "source": "overlap, parent/child, arrays, YAML edge cases"},
])
display(commitpack_setting)

split_setting = pd.DataFrame([
    {"axis": "Repository", "default": "80/10/10 repository-disjoint train/validation/test"},
    {"axis": "Content", "default": "deduplicate normalized old/new documents and operation sets"},
    {"axis": "Templates", "default": "reserve instruction templates for evaluation"},
    {"axis": "Vocabulary", "default": "report seen and held-out leaf-key results"},
    {"axis": "Oracle isolation", "default": "keep extracted final state unavailable to pseudo-label generation"},
])
display(split_setting)


### CommitPack guard ladder: illustrative cases only

These are design examples, not the output of a guard implementation. In particular, no claim is made that `jsonpatch.make_patch` provides the desired canonical diff.


In [ ]:
commitpack_guard_cases = pd.DataFrame([
    {
        "gate": "G0 — parseable concatenation",
        "expected": "accept",
        "case": "Concatenate two parseable patch lists without interaction checks.",
        "why": "Unfiltered baseline; application and interaction errors remain.",
    },
    {
        "gate": "G1 — syntax/fields",
        "expected": "reject",
        "case": '[{"op": "replace", "value": 3}]',
        "why": "The JSON Pointer path is missing.",
    },
    {
        "gate": "G1 — allowed operations",
        "expected": "reject",
        "case": '[{"op": "move", "from": "/a", "path": "/b"}]',
        "why": "Move/copy/test are outside the MVP operation set.",
    },
    {
        "gate": "G2 — individual application",
        "expected": "reject",
        "case": 'Remove /service/missing from a document where that key does not exist.',
        "why": "The component patch fails on a fresh copy of the base document.",
    },
    {
        "gate": "G3 — static conflict",
        "expected": "reject",
        "case": "One patch replaces /spec while another replaces /spec/replicas.",
        "why": "The paths have an ancestor/descendant relationship.",
    },
    {
        "gate": "G3 — static conflict",
        "expected": "reject",
        "case": "Two components add different values at /service/timeout.",
        "why": "Duplicate additions compete for the same key.",
    },
    {
        "gate": "G4 — commutativity",
        "expected": "accept",
        "case": "Replace /spec/replicas and /logging/level in either order.",
        "why": "Both orders succeed and produce the same normalized document.",
    },
    {
        "gate": "G4 — commutativity",
        "expected": "reject",
        "case": "Remove /items/0 and replace /items/1/name.",
        "why": "Array index shifting makes the final state order-dependent.",
    },
    {
        "gate": "G5 — optional schema",
        "expected": "reject",
        "case": "A patch applies but changes replicas from an integer to a string.",
        "why": "Optional domain schema validation catches a semantically invalid result.",
    },
])
display(commitpack_guard_cases)


## 3. Candidate comparison and next questions


In [ ]:
bfcl_unique_simple = int(
    bfcl_profile_df.loc[
        bfcl_profile_df["category"] == "simple",
        "unique_reference_rows",
    ].iloc[0]
)
commitpack_exact_counts = (
    valid_commitpack_df[
        valid_commitpack_df["operation_count"].isin([1, 2, 4])
    ]
    .groupby(["language", "operation_count"])
    .size()
    .rename("rows")
    .reset_index()
)
display(commitpack_exact_counts)

comparison = pd.DataFrame([
    {
        "dimension": "Atomic supervision",
        "BFCL-Comp": "One function call with BFCL possible answers",
        "CommitPack-Config": "One structural edit extracted from before/after documents",
    },
    {
        "dimension": "Composition",
        "BFCL-Comp": "Join independent requests; flatten call lists",
        "CommitPack-Config": "Join same-commit edit clauses; concatenate patches",
    },
    {
        "dimension": "Key guard interaction",
        "BFCL-Comp": "Schema validity, duplicate calls, cross-clause dependence",
        "CommitPack-Config": "Path overlap, failed application, non-commutativity",
    },
    {
        "dimension": "Primary behavioral score",
        "BFCL-Comp": "All-calls multiset exact match",
        "CommitPack-Config": "Applied final-state exact match with no collateral change",
    },
    {
        "dimension": "Main data concern exposed here",
        "BFCL-Comp": f"Only {bfcl_unique_simple} sampled/full Simple references are uniquely serializable",
        "CommitPack-Config": "Diff canonicalization, prompt faithfulness, and safe multi-edit yield",
    },
    {
        "dimension": "Natural evaluation",
        "BFCL-Comp": "Original Parallel and Parallel Multiple categories",
        "CommitPack-Config": "Filtered original commit subjects",
    },
])
display(comparison)

display(Markdown(
    f"""
### What this bounded pass establishes

- BFCL files and possible answers align by ID, but accepted-value ambiguity needs an explicit policy before producing exact SFT targets.
- The first **{COMMITPACK_SCAN_ROWS}** rows per CommitPack language contain usable one-, two-, and four-operation previews in this run; these prefix counts should not be treated as population estimates.
- Both task ideas have concrete manual `1 → 2 → 4` curricula and cheap static task structure.
- CommitPack requires substantially more preprocessing judgment before its examples become trustworthy.

### Questions for the next, serious build

1. **BFCL:** Should ambiguous possible answers be excluded, deterministically canonicalized, or represented as multiple acceptable targets?
2. **BFCL:** How much function-schema overlap is acceptable between the generated curriculum and original Parallel evaluation?
3. **CommitPack:** Which structural-diff library and canonicalization policy produce stable atomic operations?
4. **CommitPack:** Do controlled instructions faithfully describe real changes, and does a symbolic parser make them trivial?
5. **CommitPack:** What is the safe multi-edit yield after repository splitting, license filtering, strict YAML rejection, and context limits?
6. **Both:** Does a small seed model actually show the required easy-to-hard accuracy gap before any self-improvement training?
"""
))


## Environment snapshot

This final cell records the small set of package versions that affect the exploratory loading and diff preview.


In [ ]:
packages = ["requests", "pandas", "PyYAML", "jsonpatch", "matplotlib", "nbformat", "nbclient"]
version_rows = []
for package in packages:
    try:
        version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        version = "not installed"
    version_rows.append({"package": package, "version": version})
display(pd.DataFrame(version_rows))
